# BirdCLEF+ 2025 — Inference (submission notebook)

Runs under the competition limits: **CPU only, internet off**.
Loads `model.pth`, `cfg.json` and `species.json` from the training notebook's output,
splits each 1-minute test soundscape into twelve 5 s windows, and writes `submission.csv`.

The hidden test set only appears when Kaggle scores a submission. In a normal run
`test_soundscapes/` is empty, so we do a dry run on a few unlabelled training soundscapes
to check the pipeline end to end.

In [ ]:
import os, glob, json, time
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import timm

def find_data_dir():
    hits = glob.glob("/kaggle/input/**/birdclef-2025/train.csv", recursive=True) \
        or glob.glob("/kaggle/input/*/train.csv")
    return Path(hits[0]).parent if hits else Path("../../data")

def find_model_dir():
    hits = glob.glob("/kaggle/input/**/model.pth", recursive=True)
    return Path(hits[0]).parent if hits else Path("../../outputs/jamie_train_baseline")

DATA, MODEL_DIR = find_data_dir(), find_model_dir()
CFG = json.load(open(MODEL_DIR / "cfg.json"))
species = json.load(open(MODEL_DIR / "species.json"))
torch.set_num_threads(os.cpu_count())
print("Data:", DATA, "| model:", MODEL_DIR, "| CPUs:", os.cpu_count())

sample_sub = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample_sub.columns[1:]) == species, "species order differs from sample_submission"
print(sample_sub.iloc[:3, :4])

In [ ]:
test_files = sorted((DATA / "test_soundscapes").glob("*.ogg"))
DRY_RUN = len(test_files) == 0
if DRY_RUN:
    test_files = sorted((DATA / "train_soundscapes").glob("*.ogg"))[:20]
print(f"{'DRY RUN on' if DRY_RUN else 'Predicting'} {len(test_files)} soundscapes")

## Mel spectrograms (identical settings to training, read from `cfg.json`)

In [ ]:
WIN = CFG["sr"] * CFG["duration"]

def to_mel(y):
    m = librosa.feature.melspectrogram(
        y=y, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    db = librosa.power_to_db(m, ref=np.max, top_db=80)
    return ((db + 80) / 80).astype(np.float32)

def soundscape_windows(path):
    y, _ = sf.read(path, dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    mels = np.stack([to_mel(y[i * WIN:(i + 1) * WIN]) for i in range(n_win)])
    # row_id = <file stem>_<end time in seconds>, e.g. soundscape_8358733_5
    row_ids = [f"{path.stem}_{(i + 1) * CFG['duration']}" for i in range(n_win)]
    return row_ids, mels

t0 = time.time()
with Pool(os.cpu_count()) as pool:
    results = pool.map(soundscape_windows, test_files)
row_ids = [r for ids, _ in results for r in ids]
X = np.concatenate([m for _, m in results])
print("Windows:", X.shape, f"in {time.time() - t0:.0f}s")

## Predict

In [ ]:
model = timm.create_model(CFG["model"], pretrained=False, in_chans=1, num_classes=len(species))
model.load_state_dict(torch.load(MODEL_DIR / "model.pth", map_location="cpu"))
model.eval()

t0 = time.time(); preds = []
with torch.inference_mode():
    for i in range(0, len(X), 64):
        xb = torch.from_numpy(X[i:i + 64])[:, None]
        preds.append(torch.sigmoid(model(xb)).numpy())
preds = np.concatenate(preds)
print("Predictions:", preds.shape, f"in {time.time() - t0:.0f}s")

In [ ]:
sub = pd.DataFrame(preds, columns=species)
sub.insert(0, "row_id", row_ids)
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.iloc[:5, :6]